# Compose As You Are — Python Music Lab

## Google Colabスターター

ブラウザアプリで学んだ変数、リスト、`for`、`if`、乱数、関数を使い、Colab上で自分の曲を発展させます。

このノートブックでは、次の順で進みます。

1. アプリと同じ教育用作曲APIを準備する
2. Pythonコードで曲データを作る
3. 曲データを表とピアノロールで確認する
4. NumPyで簡単な音声波形を合成する
5. 独自の作曲規則やデータ可聴化へ発展させる

最初は上から順番に実行し、**「自分の曲を書く」セルだけを少しずつ変更**してください。

## 1. 作曲APIを準備する

公開サイトから`caya_music.py`を読み込みます。リポジトリをローカルで開いている場合は、手元のファイルを優先します。

In [ ]:
from pathlib import Path
import sys
import urllib.request

API_URL = "https://shogo-ishikawa.github.io/compose-as-you-are/assets/python/caya_music.py"
local_candidates = [
    Path("../assets/python/caya_music.py"),
    Path("assets/python/caya_music.py"),
    Path("caya_music.py"),
]

api_path = next((path.resolve() for path in local_candidates if path.exists()), None)

if api_path is None:
    api_path = Path("caya_music.py").resolve()
    urllib.request.urlretrieve(API_URL, api_path)
    print("公開サイトから作曲APIを読み込みました。")
else:
    print("手元の作曲APIを使います:", api_path)

sys.path.insert(0, str(api_path.parent))

## 2. 自分の曲を書く

次のコードはブラウザアプリと同じ形です。まずそのまま実行し、次に`tempo`、`seed`、`notes`、条件式などを1か所ずつ変更してください。

In [ ]:
from caya_music import *
import random

# 変更しやすい設定
tempo = 108
sound = "pluck"
seed = 7
notes = ["C4", "D4", "E4", "G4", "A4", "C5"]

random.seed(seed)

start_song(
    tempo=tempo,
    instrument=sound,
    title="Colabで作る自動作曲",
)

for i in range(24):
    # 4回ごとの最後を休符にする
    if i % 4 == 3:
        add_rest(0.5)
    else:
        note = random.choice(notes)
        velocity = random.choice([0.55, 0.70, 0.85])
        add_note(note, 0.5, velocity)

song = export_song()
print("曲名:", song["title"])
print("テンポ:", song["tempo"], "BPM")
print("長さ:", song["length_beats"], "拍")

## 3. 曲データを表で読む

Pythonコードが直接音を鳴らしているのではなく、まず音符の開始位置、長さ、音名、強さを持つ辞書データを作っています。

In [ ]:
import pandas as pd

rows = []
for track in song["tracks"]:
    for event in track["events"]:
        if event["kind"] == "note":
            label = " + ".join(event["notes"])
        else:
            label = event["sound"]
        rows.append({
            "track": track["name"],
            "instrument": track["instrument"],
            "beat": event["beat"],
            "duration": event["duration"],
            "sound": label,
            "velocity": event["velocity"],
        })

song_table = pd.DataFrame(rows)
song_table

## 4. ピアノロールを描く

横軸は時間、縦軸は音の高さです。コードの反復や条件分岐が、図の規則として見えるか確認してください。

In [ ]:
import re
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

NOTE_PATTERN = re.compile(r"^([A-Ga-g])([#b]?)(-?\d+)$")
SEMITONES = {"C": 0, "D": 2, "E": 4, "F": 5, "G": 7, "A": 9, "B": 11}


def note_to_midi(note):
    match = NOTE_PATTERN.fullmatch(note)
    if match is None:
        raise ValueError(f"音名を読み取れません: {note}")
    letter, accidental, octave_text = match.groups()
    pitch = SEMITONES[letter.upper()]
    if accidental == "#":
        pitch += 1
    elif accidental == "b":
        pitch -= 1
    octave = int(octave_text)
    return 12 * (octave + 1) + pitch


fig, ax = plt.subplots(figsize=(12, 5))
track_count = max(1, len(song["tracks"]))

for track_index, track in enumerate(song["tracks"]):
    for event in track["events"]:
        if event["kind"] != "note":
            continue
        for note in event["notes"]:
            midi = note_to_midi(note)
            rectangle = Rectangle(
                (event["beat"], midi - 0.4),
                event["duration"],
                0.8,
                alpha=0.55 + 0.35 * event["velocity"],
            )
            ax.add_patch(rectangle)

ax.set_xlim(0, max(1, song["length_beats"]))
all_midis = [
    note_to_midi(note)
    for track in song["tracks"]
    for event in track["events"]
    if event["kind"] == "note"
    for note in event["notes"]
]
if all_midis:
    ax.set_ylim(min(all_midis) - 2, max(all_midis) + 2)
ax.set_xlabel("Beat")
ax.set_ylabel("Pitch (MIDI note number)")
ax.set_title(song["title"])
ax.grid(True, alpha=0.25)
plt.show()

## 5. 曲データから音声を合成する

次のセルは、音符を単純な波形へ変換する準備です。最初は内部の数式をすべて理解する必要はありません。興味を持った場合は、周波数、包絡線、倍音、サンプリング周波数を調べてください。

In [ ]:
import math
import numpy as np
from IPython.display import Audio, display


def note_to_frequency(note):
    midi = note_to_midi(note)
    return 440.0 * 2 ** ((midi - 69) / 12)


def make_envelope(length, sample_rate, attack=0.012, release=0.10):
    envelope = np.ones(length)
    attack_samples = min(length, max(1, int(attack * sample_rate)))
    release_samples = min(length, max(1, int(release * sample_rate)))
    envelope[:attack_samples] = np.linspace(0, 1, attack_samples)
    envelope[-release_samples:] *= np.linspace(1, 0, release_samples)
    return envelope


def synthesise_song(song, sample_rate=22050):
    seconds_per_beat = 60.0 / song["tempo"]
    total_seconds = song["length_beats"] * seconds_per_beat + 0.3
    audio = np.zeros(int(total_seconds * sample_rate), dtype=np.float64)
    random_generator = np.random.default_rng(0)

    for track in song["tracks"]:
        track_gain = float(track.get("volume", 0.8))
        for event in track["events"]:
            start = int(event["beat"] * seconds_per_beat * sample_rate)
            duration_seconds = event["duration"] * seconds_per_beat
            length = max(1, int(duration_seconds * sample_rate))
            end = min(len(audio), start + length)
            length = end - start
            if length <= 0:
                continue

            t = np.arange(length) / sample_rate
            velocity = float(event["velocity"])
            envelope = make_envelope(length, sample_rate)

            if event["kind"] == "note":
                wave = np.zeros(length)
                for note in event["notes"]:
                    frequency = note_to_frequency(note)
                    fundamental = np.sin(2 * math.pi * frequency * t)
                    second_harmonic = 0.22 * np.sin(2 * math.pi * 2 * frequency * t)
                    wave += fundamental + second_harmonic
                wave /= max(1, len(event["notes"]))
            else:
                sound = event.get("sound", "hihat")
                noise = random_generator.normal(0, 1, length)
                if sound == "kick":
                    frequency = np.linspace(130, 48, length)
                    phase = 2 * math.pi * np.cumsum(frequency) / sample_rate
                    wave = np.sin(phase) * np.exp(-8 * t)
                elif sound == "snare":
                    wave = 0.75 * noise * np.exp(-13 * t)
                elif sound == "clap":
                    wave = noise * np.exp(-18 * t)
                else:
                    wave = 0.45 * noise * np.exp(-35 * t)

            audio[start:end] += track_gain * velocity * envelope * wave

    peak = np.max(np.abs(audio))
    if peak > 0:
        audio = 0.90 * audio / peak
    return audio, sample_rate


audio_data, sample_rate = synthesise_song(song)
display(Audio(audio_data, rate=sample_rate))

## 6. 小さな発展課題

次の課題から1つを選び、変更前後のコードと結果を記録してください。

- `seed`だけを3通り変え、旋律の違いを比較する
- `notes`へ同じ音を複数回入れ、選ばれる確率を変える
- `i % 4 == 3`を別の条件へ変え、休符の周期を変える
- 音の長さをリストからランダムに選ぶ
- `make_phrase(notes, repeats)`という関数を作る
- メロディにベースまたはドラムのトラックを加える

In [ ]:
# ここへ自分の発展コードを書きます。
# 最初は1か所だけ変更し、song = export_song()を最後に実行してください。

## 7. データを音へ変換する例

数値データを、音の高さの番号へ変換します。下の温度は模擬データです。自分の学科や関心に合うデータへ置き換えてください。

In [ ]:
from caya_music import *

values = [18, 19, 21, 24, 27, 29, 28, 25, 22, 20]
scale = ["C4", "D4", "E4", "G4", "A4", "C5"]

start_song(
    tempo=92,
    instrument="warm_pad",
    title="数値データの可聴化",
)

minimum = min(values)
maximum = max(values)
width = maximum - minimum

for value in values:
    ratio = (value - minimum) / width
    index = round(ratio * (len(scale) - 1))
    note = scale[index]
    add_note(note, 0.75, 0.68)
    print(value, "→", note)

data_song = export_song()
data_audio, data_rate = synthesise_song(data_song)
display(Audio(data_audio, rate=data_rate))

## 振り返り

次の文章を完成させてください。

- 私が使ったPython文法は、＿＿＿＿＿＿＿＿である。
- ＿＿行目の＿＿＿＿を変更すると、音は＿＿＿＿＿＿＿＿と変化した。
- この自動作曲で人が決めた規則は、＿＿＿＿＿＿＿＿である。
- 次に試したい改善は、＿＿＿＿＿＿＿＿である。